

# M3 2 Beautiful Soup Model Updates
### OPIM 5512 — Applied Data Science · Module3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/M3_2_BeautifulSoup_ModelUpdates.ipynb)

*Run me top to bottom — **Runtime → Run all**. Data loads from a stable link, so there's nothing to upload.*

# Understanding and advancing the "traditional" RegEx/ETL code
---------------------
**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**


Reference on RegEx:
* [Overview of Regular Expressions (RegEx) from Microsoft](https://learn.microsoft.com/en-us/dotnet/standard/base-types/regular-expression-language-quick-reference)

# Instructions

The ETL step is essentially takes your raw txt file and makes a clean json file.

```
{
  "post_id": "1234567890",
  "run_id": "20251026170002",
  "scraped_at": "2025-10-26T17:00:02Z",
  "source_txt": "scrapes/run_id=20251026170002/1234567890.txt",
  "price": 12995,
  "year": 2015,
  "make": "Honda",
  "model": "Civic",
  "mileage": 87500
}
```

These are the fields that are going to be populated. Those last fields (`price, year, make, model, mileage`) are what you’ll later use for modeling.

The important question is: how does `parse_listing()` find those fields from plain text?

**Answer:** regular expressions (regex).

# Three main regular expressions... what do they do?
Near the top of `main.py` you will see...

```
PRICE_RE      = re.compile(r"\$\s?([0-9,]+)")
YEAR_RE       = re.compile(r"\b(19|20)\d{2}\b")
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
```

##🔹 PRICE_RE – find the price

`PRICE_RE = re.compile(r"\$\s?([0-9,]+)")`

This looks for patterns like:

* `$12995`
* `$ 12,995`
* `$ 7,500`

Pieces:
* `\$` – a literal dollar sign.
* `\s?` – optional whitespace after it (space or no space).
* `([0-9,]+)` – capture group of one or more digits or commas (e.g., 12,995)

In `parse_listing()`:

```
m = PRICE_RE.search(text)
if m:
    d["price"] = int(m.group(1).replace(",", ""))
```

1. `PRICE_RE.search(text)` finds the first **$…** price in the text.
2. `m.group(1)` is the part inside parentheses → `12,995`.
3. `.replace(",", "")` cleans it up → "12995".
4. `int(...)` converts to a numeric/integer → 12995.

Result: `d["price"] = 12995`


**On your own:** Go read two or three listings to confirm this is how it actually works!

## 🔹 YEAR_RE – find the year of the car

`YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")`

This looks for 4-digit years starting with 19 or 20, e.g.:
* 1999
* 2008
* 2015
* 2022

Pieces:
* `\b` – word boundary (start/end of a word).
* `(19|20)` – either 19 or 20.
* `\d{2}` – two more digits (so total 4 digits).
* `\b` – another word boundary.

In `parse_listing()`:

```
y = YEAR_RE.search(text)
if y:
    d["year"] = int(y.group(0))
```

* `y.group(0)` is the whole match, like "2015".
* Turn it into an integer and store as `year`.

**On your own:** Go read two or three listings to confirm this is how it actually works! What are some limitations?

## 🔹 MAKE_MODEL_RE – guess make and model

`MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
`

This is a simple heuristic to grab:
* a capitalized word (make), followed by
* a capitalized word (model),

like:
* Honda Civic
* Toyota Camry
* Ford F150
*... Contact Information 😃... oops! Students can fix this!

Pieces:
* `\b` – word boundary.
* `([A-Z][a-z]+)` – **first group**: one capital letter followed by one or more lowercase letters. That matches “Honda”, “Toyota”, “Ford”.
* `\s+` – one or more spaces.
* `([A-Z][A-Za-z0-9]+)` – **second group**: capital letter followed by letters or numbers. That matches “Civic”, “Camry”, “F150”.


In `parse_listing()`:

```
mm = MAKE_MODEL_RE.search(text)
if mm:
    d["make"]  = mm.group(1)  # e.g. "Honda"
    d["model"] = mm.group(2)  # e.g. "Civic"
```

So the first matching “Title Case pair” is interpreted as `make` and `model`.
It’s not perfect, but often works well enough for homework and first attempt.

**On Your Own:** How can we strengthen the logic in `make` or `model`? Do we need a lookup table?

# Mileage: with three flavors

Mileage is trickier because people write it in lots of ways. The code uses 3 attempts in this order:

```python
mi = None
m1 = re.search(r"(?:mileage|odometer)\s*[:\-]?\s*([\d,]+)", text, re.I)
...
m2 = re.search(r"(\d+(?:\.\d+)?)\s*k\s*(?:mi|mile|miles)\b", text, re.I)
...
m3 = re.search(r"(\d{1,3}(?:[,\d]{3})*)\s*(?:mi|mile|miles)\b", text, re.I)
```

Let’s decode each.

## 🔹 First attempt: explicit “mileage” or “odometer” label

```python
m1 = re.search(r"(?:mileage|odometer)\s*[:\-]?\s*([\d,]+)", text, re.I)
```

This matches things like:
* mileage: 123,456
* odometer 87500
* mileage- 90,000

Pieces:
* `(?:mileage|odometer)` – non-capturing group for either the word “mileage” or “odometer”.
* `\s*` – any spaces.
* `[:\-]?` – optional colon or dash.
* `\s*` – more spaces.
* `([\d,]+)` – capture digits/commas as the mileage value.

`re.I` means case-insensitive (so “Mileage”, “mileage”, “MILEAGE” all match).

If we find it:
```python
if m1:
    mi = int(m1.group(1).replace(",", ""))
```

So `m1.group(1)` is `"123,456"` → becomes `123456`

## 🔹 Second attempt: “Xk miles” style

```python
m2 = re.search(r"(\d+(?:\.\d+)?)\s*k\s*(?:mi|mile|miles)\b", text, re.I)
```

Matches things like:
* 80k mi
* 65.5k miles
* 50k mile

Pieces:
* `(\d+(?:\.\d+)?)` – capture a number, maybe with a decimal: 80 or 65.5.
* `\s*k\s*` – letter k (thousand), with optional spaces around it.
* `(?:mi|mile|miles)` – non-capturing group: any of mi, mile, or miles.

If THIS matches, we convert it to a full integer:
```python
mi = int(float(m2.group(1)) * 1000)
```

So:
* `80k → 80 * 1000 = 80000`
* `65.5k → 65.5 * 1000 ≈ 65500`

## 🔹 Third attempt: plain “X miles” style

```python
m3 = re.search(r"(\d{1,3}(?:[,\d]{3})*)\s*(?:mi|mile|miles)\b", text, re.I)
```

Matches things like:
* 120,000 miles
* 98,500 mi
* 75,000 mile

Pieces:
* `(\d{1,3}(?:[,\d]{3})*)` – start with 1–3 digits, then allow repeated `,ddd` groups.
  * This matches stuff like 120,000, 98,500, 75,000.
* `\s*` – optional spaces.
* `(?:mi|mile|miles)` – mi, mile, or miles.

Then:
```python
mi = int(re.sub(r"[^\d]", "", m3.group(1)))
```

It strips the commas or other non-digits and converts to an int.

**On Your Own:** Can you think of OTHER ways that we might be able to better capture mileage? Other fields like `paint color` that you might be able to add?

# On Your Own

## Compute the new field in `parse_listing`

This is the main ETL brain:

```python
def parse_listing(text: str) -> dict:
    d = {}

    # price, year, make, model, mileage logic...

    return d
```

If a student wants to add, say, `num_doors` (2 or 4... or more?!), `fuel_type` (diesel or gasoline or electric), `transmission` (automatic or manual), or a simple flag like is_truck, they:
* add new regex / parsing logic inside parse_listing
* set `d["new_field_name"] = some_value`

Because `extract_http` later does:

```python
fields = parse_listing(text)

record = {
    "post_id": post_id,
    "run_id": run_id,
    "scraped_at": scraped_at_iso,
    "source_txt": name,
    **fields,
}
```

anything they add to `d` in `parse_listing` will automatically show up in the final JSON.

So the right place for new features is `parse_listing()`. If you had CONSTANTS that aren't in the text, we could just add some stuff to `record` directly.  But `parse_listing` makes the most sense!

## Write a new `materialize-etl` function and deploy to GCP!
You should anticipate that, if you add new fields, your materialize function is going to BREAK! That's because the number of columns won't match when you concatenate/append everything.

However, you should be able to hack your existing materialize cloud function... and make a copy of it called `materialize-v2` and start aggregating all of the new LLM data you are creating. You can have the job run once and hour or once a day. The most important thing is that you have growing record of the new RegEx-ETL outputs.

There are many ways to do this - maybe only read files after a certain date (when you made the change) - or maybe only read files that have the new column names, otherwise ignore. You can choose the best way for you.


## Make some cool outputs to show-off your skills
Make new jobs/functions that take your `materialize` data and track things like:
* Number of new cars for sale per day
* Avg price of car per day
* Percentage of electric cars vs. fossil-fuel powered cars per day

...and if you want, try to post the outputs back on your GitHub or some dashboard!